In [0]:
%sql
SELECT * FROM brazilian_ecommerce_dataset_olist.brazilian_ecommerce_dataset_olist_schema.silver_orders_enriched


In [0]:
%sql

-- How many orders were successfully delivered?
-- How many orders arrived late?
-- What percentage of delivered orders were late?
-- How many days does delivery take on average?
-- What is the average customer review score?
CREATE OR REPLACE TABLE brazilian_ecommerce_dataset_olist.brazilian_ecommerce_dataset_olist_schema.gold_delivery_kpis USING DELTA AS
SELECT count(*) AS total_delivered,
SUM(is_late) AS late_orders,
SUM(is_late)/count(*) * 100.0 AS pct_late_pct,
ROUND(AVG(delivery_days),2) AS avg_delivery_days,
ROUND(AVG(review_score), 2) AS avg_review_score
FROM brazilian_ecommerce_dataset_olist.brazilian_ecommerce_dataset_olist_schema.silver_orders_enriched
WHERE is_delivered = true;



In [0]:
%sql 
SELECT * FROM brazilian_ecommerce_dataset_olist.brazilian_ecommerce_dataset_olist_schema.gold_delivery_kpis

In [0]:
%sql
-- How many orders were delivered in each state?
-- How many were late in each state?
-- Which states have the highest late-delivery percentages?
-- How long does delivery take on average in each state?

CREATE OR REPLACE TABLE brazilian_ecommerce_dataset_olist.brazilian_ecommerce_dataset_olist_schema.gold_delivery_state_kpis USING DELTA AS
SELECT customer_state,
COUNT(*) AS total_delivered_in_each_state,
SUM(is_late) AS late_orders_in_each_state,
SUM(is_late)/COUNT(*) * 100.0 AS pct_late_pct,
ROUND(AVG(delivery_days),2) AS avg_delivery_days
FROM brazilian_ecommerce_dataset_olist.brazilian_ecommerce_dataset_olist_schema.silver_orders_enriched
WHERE is_delivered = true AND customer_state IS NOT NULL
GROUP BY customer_state 
ORDER BY pct_late_pct DESC;

In [0]:
%sql

SELECT * FROM brazilian_ecommerce_dataset_olist.brazilian_ecommerce_dataset_olist_schema.gold_delivery_state_kpis

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- How many orders arrived late versus on time?
-- How many orders in each group have review scores?
-- What is the average review score for late versus on-time deliveries?
-- What percentage of reviewed orders received low ratings (1 or 2) in each group?
CREATE OR REPLACE TABLE brazilian_ecommerce_dataset_olist.brazilian_ecommerce_dataset_olist_schema.gold_review_kpis USING DELTA AS
SELECT is_late,
COUNT(*) AS total_orders,
COUNT(review_score) AS total_review_counts,
ROUND(AVG(review_score),2) AS avg_scores,
SUM(CASE WHEN review_score <= 2 THEN 1 ELSE 0 END)/COUNT(review_score) * 100.0 AS pct_low_ratings
FROM brazilian_ecommerce_dataset_olist.brazilian_ecommerce_dataset_olist_schema.silver_orders_enriched
WHERE is_delivered = true
GROUP BY is_late;

In [0]:
%sql
SELECT * FROM brazilian_ecommerce_dataset_olist.brazilian_ecommerce_dataset_olist_schema.gold_review_kpis 

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- How many delivered orders belong to each primary product category?
-- How many orders were late in each category?
-- Which categories have the highest late-delivery percentages?
-- What is the average review score for each category?

CREATE OR REPLACE TABLE brazilian_ecommerce_dataset_olist.brazilian_ecommerce_dataset_olist_schema.gold_category_kpis USING DELTA AS
SELECT 
COALESCE(product_category_name_english, primary_category, 'Unknown') AS category,
COUNT(*) AS total_delivered,
SUM(is_late) AS late_orders,
ROUND(SUM(is_late)/COUNT(*) * 100.0, 2) AS late_delivery_pct,
ROUND(AVG(review_score), 2) AS avg_review_score
FROM brazilian_ecommerce_dataset_olist.brazilian_ecommerce_dataset_olist_schema.silver_orders_enriched
WHERE is_delivered = true
GROUP BY COALESCE(product_category_name_english, primary_category, 'Unknown')
HAVING COUNT(*) >= 20
ORDER BY late_delivery_pct DESC;

In [0]:
%sql
SELECT * FROM brazilian_ecommerce_dataset_olist.brazilian_ecommerce_dataset_olist_schema.gold_category_kpis

In [0]:
%sql
-- How many orders purchased in each month were eventually delivered?
-- What percentage of those orders arrived late?
-- What was the average delivery duration for orders purchased in each month?
-- Are there months with particularly poor delivery performance?
CREATE OR REPLACE TABLE brazilian_ecommerce_dataset_olist.brazilian_ecommerce_dataset_olist_schema.gold_monthly_kpis USING DELTA AS
SELECT purchase_month,
COUNT(*) AS total_delivered,
SUM(is_late)/COUNT(*) * 100.0 AS late_delivery_pct,
ROUND(AVG(delivery_days),2) AS avg_delivery_days
FROM brazilian_ecommerce_dataset_olist.brazilian_ecommerce_dataset_olist_schema.silver_orders_enriched
WHERE is_delivered = true AND purchase_month IS NOT NULL
GROUP BY purchase_month
HAVING total_delivered > 20
ORDER BY late_delivery_pct DESC;

In [0]:
%sql

SELECT * FROM brazilian_ecommerce_dataset_olist.brazilian_ecommerce_dataset_olist_schema.gold_monthly_kpis 
ORDER BY late_delivery_pct DESC;

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.